# 🏒 Hockey Player Tracking — Phase 0 Benchmark

**What this notebook does:**
1. Clones the `hockey-tracking` repo and installs all dependencies
2. Creates synthetic broadcast hockey data (or loads VIP-HTD if available)
3. Runs RF-DETR detection and measures recall
4. Benchmarks every available tracker (SORT → ByteTrack → OC-SORT → BoT-SORT → ...)
5. Evaluates all trackers with TrackEval (HOTA, IDF1, MOTA, IDsw)
6. Produces a single comparison table

**Runtime:** ~10 min on T4 with synthetic data, longer with real video.

---

## 1. Setup

In [ ]:
# Clone the repo (change to your fork if needed)
!git clone https://github.com/FirebladeFN/hockey-tracking.git 2>/dev/null || echo 'Repo already cloned'
%cd hockey-tracking

In [ ]:
# Install all dependencies
!bash scripts/setup_colab.sh

In [ ]:
# Verify imports
import sys, os
sys.path.insert(0, '.')

from src.detection.data_loader import load_dataset, load_sequence, load_mot_txt, write_mot_txt, Detection
from eval.detection_eval import evaluate_detections, print_results_table, compute_iou
from src.tracking.benchmark_trackers import TRACKER_REGISTRY, KNOWN_TRACKERS, list_trackers

print('All core imports OK')
list_trackers()

## 2. Dataset

**Option A:** Use synthetic data (always works, for pipeline validation)

**Option B:** Load VIP-HTD or your own broadcast hockey clips into `data/vip_htd/`

Toggle `USE_REAL_DATA` below.

In [ ]:
USE_REAL_DATA = False  # Set True if you have VIP-HTD in data/vip_htd/

if USE_REAL_DATA:
    DATA_DIR = 'data/vip_htd'
    SPLIT = 'train'
    !python scripts/verify_vip_htd.py --base_dir {DATA_DIR}
else:
    # Create synthetic broadcast hockey data
    import random, glob
    from pathlib import Path
    import numpy as np
    import cv2

    DATA_DIR = 'data/synthetic'
    SPLIT = 'train'
    NUM_SEQUENCES = 3
    NUM_FRAMES = 100
    NUM_OBJECTS = 8  # ~realistic for broadcast hockey (visible players)

    random.seed(42)
    np.random.seed(42)

    for seq_idx in range(1, NUM_SEQUENCES + 1):
        seq_name = f'seq_{seq_idx:03d}'
        seq_dir = Path(DATA_DIR) / SPLIT / seq_name
        img_dir = seq_dir / 'img1'
        gt_dir = seq_dir / 'gt'
        img_dir.mkdir(parents=True, exist_ok=True)
        gt_dir.mkdir(parents=True, exist_ok=True)

        # Create realistic-ish synthetic frames (ice rink colors)
        gt_lines = []
        W, H = 1920, 1080

        # Initialize player positions and velocities
        positions = []
        velocities = []
        for obj_id in range(NUM_OBJECTS):
            x = random.uniform(200, W - 200)
            y = random.uniform(150, H - 150)
            vx = random.uniform(-15, 15)  # pixels/frame
            vy = random.uniform(-8, 8)
            positions.append([x, y])
            velocities.append([vx, vy])

        for frame in range(1, NUM_FRAMES + 1):
            # Create frame image (white ice + blue lines)
            img = np.full((H, W, 3), (240, 240, 245), dtype=np.uint8)  # ice white
            # Red center line
            cv2.line(img, (W//2, 0), (W//2, H), (0, 0, 200), 3)
            # Blue lines
            cv2.line(img, (W//3, 0), (W//3, H), (200, 100, 0), 2)
            cv2.line(img, (2*W//3, 0), (2*W//3, H), (200, 100, 0), 2)

            for obj_id in range(NUM_OBJECTS):
                # Update position with velocity + noise
                positions[obj_id][0] += velocities[obj_id][0] + random.uniform(-3, 3)
                positions[obj_id][1] += velocities[obj_id][1] + random.uniform(-2, 2)

                # Bounce off walls
                if positions[obj_id][0] < 50 or positions[obj_id][0] > W - 100:
                    velocities[obj_id][0] *= -1
                if positions[obj_id][1] < 50 or positions[obj_id][1] > H - 150:
                    velocities[obj_id][1] *= -1

                # Random velocity changes (hockey players change direction)
                if random.random() < 0.05:
                    velocities[obj_id][0] = random.uniform(-15, 15)
                    velocities[obj_id][1] = random.uniform(-8, 8)

                x = max(10, min(W - 60, positions[obj_id][0]))
                y = max(10, min(H - 140, positions[obj_id][1]))
                w = 45 + random.uniform(-5, 5)
                h = 120 + random.uniform(-10, 10)

                # Draw player rectangle on frame
                color = (0, 0, 180) if obj_id < NUM_OBJECTS // 2 else (180, 0, 0)
                cv2.rectangle(img, (int(x), int(y)), (int(x+w), int(y+h)), color, -1)
                cv2.rectangle(img, (int(x), int(y)), (int(x+w), int(y+h)), (0,0,0), 1)

                gt_lines.append(
                    f'{frame},{obj_id+1},{x:.2f},{y:.2f},{w:.2f},{h:.2f},1.0000,1,1.0,-1'
                )

            cv2.imwrite(str(img_dir / f'{frame:06d}.jpg'), img)

        # Write GT
        with open(gt_dir / 'gt.txt', 'w') as f:
            f.write('\n'.join(gt_lines))

        # Write seqinfo
        with open(seq_dir / 'seqinfo.ini', 'w') as f:
            f.write(f'[Sequence]\nname={seq_name}\nimDir=img1\nframeRate=30\n'
                    f'seqLength={NUM_FRAMES}\nimWidth={W}\nimHeight={H}\nimExt=.jpg\n')

    print(f'Created {NUM_SEQUENCES} synthetic sequences with {NUM_FRAMES} frames each')
    !python scripts/verify_vip_htd.py --base_dir {DATA_DIR}

In [ ]:
# Preview a frame
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

seqs = sorted((Path(DATA_DIR) / SPLIT).iterdir())
if seqs:
    frames = sorted((seqs[0] / 'img1').glob('*.jpg'))[:1]
    if frames:
        img = cv2.imread(str(frames[0]))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Overlay GT boxes
        seq = load_sequence(str(seqs[0]))
        frame1_gt = [d for d in seq.gt if d.frame_id == 1]
        for d in frame1_gt:
            cv2.rectangle(img_rgb, (int(d.x), int(d.y)),
                          (int(d.x + d.w), int(d.y + d.h)), (0, 255, 0), 2)
            cv2.putText(img_rgb, f'ID:{d.track_id}', (int(d.x), int(d.y) - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        plt.figure(figsize=(14, 8))
        plt.imshow(img_rgb)
        plt.title(f'{seqs[0].name} — Frame 1 with GT boxes ({len(frame1_gt)} objects)')
        plt.axis('off')
        plt.show()

## 3. Detection (RF-DETR)

Run RF-DETR on all frames and measure recall against GT.

In [ ]:
import time
from pathlib import Path

DET_DIR = f'eval/detections/rfdetr_coco/{SPLIT}'
os.makedirs(DET_DIR, exist_ok=True)

# Check if RF-DETR is available
try:
    from rfdetr import RFDETRBase
    USE_RFDETR = True
    print('RF-DETR available — running real inference')
except ImportError:
    USE_RFDETR = False
    print('RF-DETR not available — generating synthetic detections from GT')

gt_split = Path(DATA_DIR) / SPLIT
sequences = sorted([p.name for p in gt_split.iterdir() if p.is_dir()])
print(f'Sequences to process: {sequences}')

In [ ]:
if USE_RFDETR:
    # Real RF-DETR inference
    from src.detection.rfdetr_detector import load_detector, detect_sequence
    model = load_detector('base')

    for seq_name in sequences:
        frame_dir = str(gt_split / seq_name / 'img1')
        out_file = f'{DET_DIR}/{seq_name}.txt'
        print(f'\nDetecting: {seq_name}')
        t0 = time.time()
        n_dets, n_frames = detect_sequence(
            model, frame_dir, out_file, confidence_threshold=0.5
        )
        elapsed = time.time() - t0
        print(f'  {n_dets} detections in {n_frames} frames ({elapsed:.1f}s, {n_frames/elapsed:.1f} FPS)')
else:
    # Synthetic detections: perturb GT with realistic noise
    import random
    random.seed(123)

    for seq_name in sequences:
        seq = load_sequence(str(gt_split / seq_name))
        det_lines = []

        for d in seq.gt:
            if random.random() < 0.05:  # 5% miss rate
                continue
            noise = 8.0
            x = d.x + random.uniform(-noise, noise)
            y = d.y + random.uniform(-noise, noise)
            w = d.w + random.uniform(-noise/2, noise/2)
            h = d.h + random.uniform(-noise/2, noise/2)
            conf = max(0.1, min(1.0, random.uniform(0.4, 1.0)))
            det_lines.append(f'{d.frame_id},-1,{x:.2f},{y:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1')

        # Add false positives (~1 per 5 frames)
        max_frame = max(d.frame_id for d in seq.gt)
        for _ in range(max_frame // 5):
            frame = random.randint(1, max_frame)
            x = random.uniform(0, 1800)
            y = random.uniform(0, 900)
            w = random.uniform(30, 70)
            h = random.uniform(80, 140)
            conf = random.uniform(0.2, 0.55)
            det_lines.append(f'{frame},-1,{x:.2f},{y:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1')

        out_file = f'{DET_DIR}/{seq_name}.txt'
        with open(out_file, 'w') as f:
            f.write('\n'.join(det_lines))
        print(f'{seq_name}: {len(det_lines)} detections (synthetic)')

print('\nDetection complete.')

In [ ]:
# Evaluate detection recall
!python eval/detection_eval.py \
    --gt_dir {DATA_DIR} \
    --det_dir eval/detections/rfdetr_coco \
    --split {SPLIT} \
    --save_json eval/results/detection_eval.json

## 4. Multi-Tracker Benchmark

Run every available tracker on the same frozen detections.

In [ ]:
# Discover which trackers are actually available
available_trackers = []

# SORT
try:
    from trackers import SORTTracker
    available_trackers.append('sort')
    print('✓ SORT (roboflow/trackers)')
except ImportError:
    print('✗ SORT — pip install trackers')

# ByteTrack
try:
    from trackers import ByteTrackTracker
    available_trackers.append('bytetrack')
    print('✓ ByteTrack (roboflow/trackers)')
except ImportError:
    try:
        import supervision as sv
        _ = sv.ByteTrack
        available_trackers.append('bytetrack')
        print('✓ ByteTrack (supervision)')
    except (ImportError, AttributeError):
        print('✗ ByteTrack — pip install trackers OR pip install supervision')

# OC-SORT
try:
    from ocsort.ocsort import OCSort
    available_trackers.append('ocsort')
    print('✓ OC-SORT')
except ImportError:
    print('✗ OC-SORT — pip install ocsort')

# BoT-SORT
try:
    from ultralytics.trackers.bot_sort import BOTSORT
    available_trackers.append('botsort')
    print('✓ BoT-SORT (ultralytics)')
except ImportError:
    print('✗ BoT-SORT — pip install ultralytics')

print(f'\nAvailable trackers: {available_trackers}')
if not available_trackers:
    print('No tracker packages found! Installing fallback...')
    !pip install -q supervision
    available_trackers = ['bytetrack']

In [ ]:
import time
import json
from collections import defaultdict
from src.tracking.benchmark_trackers import TRACKER_REGISTRY, run_tracker_on_sequence

BENCHMARK_DIR = 'eval/tracker_benchmark'
os.makedirs(BENCHMARK_DIR, exist_ok=True)

benchmark_results = {}

for tracker_name in available_trackers:
    print(f'\n{"=" * 60}')
    print(f'RUNNING: {tracker_name}')
    print(f'{"=" * 60}')

    TrackerClass = TRACKER_REGISTRY[tracker_name]
    tracker_out_dir = Path(BENCHMARK_DIR) / tracker_name / SPLIT
    tracker_out_dir.mkdir(parents=True, exist_ok=True)

    seq_results = {}

    for seq_name in sequences:
        det_file = f'{DET_DIR}/{seq_name}.txt'
        seq_dir = str(gt_split / seq_name)
        out_file = str(tracker_out_dir / f'{seq_name}.txt')

        try:
            tracker = TrackerClass()
            n_tracks, elapsed = run_tracker_on_sequence(
                tracker, det_file, seq_dir, out_file
            )
            seq_info = load_sequence(seq_dir)
            fps = seq_info.frame_count / elapsed if elapsed > 0 else 0
            seq_results[seq_name] = {
                'tracks': n_tracks, 'time_s': round(elapsed, 3),
                'fps': round(fps, 1), 'status': 'ok'
            }
            print(f'  {seq_name}: {n_tracks} track rows, {elapsed:.2f}s ({fps:.0f} FPS)')
        except Exception as e:
            seq_results[seq_name] = {'status': 'error', 'error': str(e)}
            print(f'  {seq_name}: FAILED — {e}')

    benchmark_results[tracker_name] = seq_results

# Save
with open(f'{BENCHMARK_DIR}/benchmark_summary.json', 'w') as f:
    json.dump(benchmark_results, f, indent=2)

print(f'\n\nBenchmark complete. Results saved to {BENCHMARK_DIR}/')

## 5. Evaluation (TrackEval)

Run HOTA / IDF1 / MOTA / IDsw / Frag for every tracker.

In [ ]:
try:
    import trackeval
    HAS_TRACKEVAL = True
    print('TrackEval available')
except ImportError:
    HAS_TRACKEVAL = False
    print('TrackEval not available — skipping formal eval')
    print('Install with: pip install git+https://github.com/JonathonLuiten/TrackEval.git')

In [ ]:
if HAS_TRACKEVAL:
    from eval.trackeval_wrapper import stage_for_trackeval
    import trackeval

    eval_results = {}

    for tracker_name in available_trackers:
        tracker_pred_dir = f'{BENCHMARK_DIR}/{tracker_name}/{SPLIT}'
        if not os.path.exists(tracker_pred_dir):
            continue

        pred_files = list(Path(tracker_pred_dir).glob('*.txt'))
        if not pred_files:
            continue

        print(f'\nEvaluating: {tracker_name}')

        workspace = f'eval/trackeval_workspace/{tracker_name}'
        output_dir = f'eval/results/{tracker_name}'

        try:
            staging = stage_for_trackeval(
                gt_split_dir=str(gt_split),
                tracker_pred_dir=tracker_pred_dir,
                workspace_dir=workspace,
                dataset_name='HockeyBench',
                split=SPLIT,
                tracker_name=tracker_name,
                sequences=sequences,
            )

            eval_config = trackeval.Evaluator.get_default_eval_config()
            eval_config['USE_PARALLEL'] = False
            eval_config['PRINT_RESULTS'] = True
            eval_config['PLOT_CURVES'] = False

            ds_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
            ds_config['GT_FOLDER'] = staging['gt_root']
            ds_config['TRACKERS_FOLDER'] = staging['trackers_root']
            ds_config['SEQMAP_FOLDER'] = staging['seqmap_root']
            ds_config['TRACKERS_TO_EVAL'] = [tracker_name]
            ds_config['BENCHMARK'] = 'HockeyBench'
            ds_config['SPLIT_TO_EVAL'] = SPLIT
            ds_config['OUTPUT_FOLDER'] = output_dir
            ds_config['TRACKER_SUB_FOLDER'] = 'data'
            ds_config['OUTPUT_SUB_FOLDER'] = ''
            ds_config['DO_PREPROC'] = False

            evaluator = trackeval.Evaluator(eval_config)
            dataset = trackeval.datasets.MotChallenge2DBox(ds_config)
            metrics = [
                trackeval.metrics.HOTA(),
                trackeval.metrics.CLEAR(),
                trackeval.metrics.Identity(),
            ]

            results, _ = evaluator.evaluate([dataset], metrics)
            eval_results[tracker_name] = results

        except Exception as e:
            print(f'  TrackEval failed for {tracker_name}: {e}')
            eval_results[tracker_name] = {'error': str(e)}

    print('\nEvaluation complete.')
else:
    eval_results = {}
    print('Skipping TrackEval (not installed)')

## 6. Results — Comparison Table

In [ ]:
import pandas as pd
import numpy as np

# Build comparison table
rows = []

for tracker_name in available_trackers:
    row = {'Tracker': tracker_name}

    # Speed from benchmark
    if tracker_name in benchmark_results:
        seqs = benchmark_results[tracker_name]
        ok_seqs = [v for v in seqs.values() if isinstance(v, dict) and v.get('status') == 'ok']
        if ok_seqs:
            row['Avg FPS'] = round(np.mean([s['fps'] for s in ok_seqs]), 1)
            row['Total Tracks'] = sum(s['tracks'] for s in ok_seqs)

    # Metrics from TrackEval
    if tracker_name in eval_results and 'error' not in eval_results[tracker_name]:
        try:
            res = eval_results[tracker_name]
            # Navigate TrackEval's nested output
            mot_key = next(iter(res.keys()))
            tracker_block = res[mot_key][tracker_name]
            combined = tracker_block.get('COMBINED_SEQ', {})
            if combined:
                cls_key = next(iter(combined.keys()))
                metrics = combined[cls_key]
                # HOTA returns array of values at different thresholds; take mean
                hota_val = metrics.get('HOTA')
                if hasattr(hota_val, '__len__'):
                    row['HOTA'] = round(float(np.mean(hota_val)) * 100, 1)
                elif hota_val is not None:
                    row['HOTA'] = round(float(hota_val) * 100, 1)

                for metric_name in ['IDF1', 'MOTA', 'MOTP']:
                    val = metrics.get(metric_name)
                    if val is not None:
                        row[metric_name] = round(float(val) * 100, 1)

                for metric_name in ['IDSW', 'Frag']:
                    val = metrics.get(metric_name)
                    if val is not None:
                        row[metric_name] = int(val)

                assa_val = metrics.get('AssA')
                if hasattr(assa_val, '__len__'):
                    row['AssA'] = round(float(np.mean(assa_val)) * 100, 1)
                elif assa_val is not None:
                    row['AssA'] = round(float(assa_val) * 100, 1)
        except Exception as e:
            print(f'Warning: could not parse metrics for {tracker_name}: {e}')

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    # Reorder columns
    desired_cols = ['Tracker', 'HOTA', 'IDF1', 'MOTA', 'AssA', 'IDSW', 'Frag', 'Avg FPS', 'Total Tracks']
    cols = [c for c in desired_cols if c in df.columns]
    df = df[cols]

    print('\n' + '=' * 80)
    print('TRACKER COMPARISON TABLE')
    print('=' * 80)
    print(df.to_markdown(index=False))
    print()

    # Save
    df.to_csv(f'{BENCHMARK_DIR}/comparison_table.csv', index=False)
    print(f'Saved to {BENCHMARK_DIR}/comparison_table.csv')
else:
    print('No results to display.')

## 7. Visualization — Track Quality

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import numpy as np

def visualize_tracks(seq_dir, track_file, title='', max_frames=5):
    """Visualize tracked boxes on frames."""
    seq = load_sequence(seq_dir)
    tracks = load_mot_txt(track_file)

    frames_dir = Path(seq.frame_dir)
    frame_files = sorted(frames_dir.glob('*.jpg'))[:max_frames]

    if not frame_files:
        print(f'No frames found in {frames_dir}')
        return

    # Color map for track IDs
    unique_ids = sorted(set(t.track_id for t in tracks))
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_ids), 1)))
    id_to_color = {tid: colors[i % len(colors)] for i, tid in enumerate(unique_ids)}

    fig, axes = plt.subplots(1, len(frame_files), figsize=(6 * len(frame_files), 6))
    if len(frame_files) == 1:
        axes = [axes]

    for ax, fp in zip(axes, frame_files):
        frame_id = int(fp.stem)
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)

        frame_tracks = [t for t in tracks if t.frame_id == frame_id]
        for t in frame_tracks:
            color = id_to_color.get(t.track_id, (1, 0, 0, 1))
            rect = mpatches.Rectangle(
                (t.x, t.y), t.w, t.h,
                linewidth=2, edgecolor=color[:3], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(t.x, t.y - 3, f'{t.track_id}',
                    fontsize=8, color=color[:3], weight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.7))

        ax.set_title(f'Frame {frame_id} ({len(frame_tracks)} tracks)', fontsize=10)
        ax.axis('off')

    plt.suptitle(title, fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()


# Visualize first available tracker on first sequence
for tracker_name in available_trackers:
    track_file = f'{BENCHMARK_DIR}/{tracker_name}/{SPLIT}/{sequences[0]}.txt'
    if os.path.exists(track_file):
        visualize_tracks(
            str(gt_split / sequences[0]),
            track_file,
            title=f'{tracker_name.upper()} tracks on {sequences[0]}',
            max_frames=5,
        )
        break

In [ ]:
# Trajectory plot — show all tracks across frames
def plot_trajectories(track_file, title=''):
    tracks = load_mot_txt(track_file)
    if not tracks:
        print('No tracks to plot')
        return

    unique_ids = sorted(set(t.track_id for t in tracks))
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_ids), 1)))
    id_to_color = {tid: colors[i % len(colors)] for i, tid in enumerate(unique_ids)}

    plt.figure(figsize=(14, 8))
    for tid in unique_ids:
        tid_tracks = sorted([t for t in tracks if t.track_id == tid], key=lambda t: t.frame_id)
        xs = [t.x + t.w / 2 for t in tid_tracks]
        ys = [t.y + t.h / 2 for t in tid_tracks]
        plt.plot(xs, ys, '-o', color=id_to_color[tid], markersize=2, linewidth=1.5, label=f'ID {tid}')

    plt.gca().invert_yaxis()
    plt.xlabel('X (pixels)')
    plt.ylabel('Y (pixels)')
    plt.title(title)
    if len(unique_ids) <= 15:
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f'{len(unique_ids)} unique track IDs, {len(tracks)} total track rows')


# Compare trajectories across trackers
for tracker_name in available_trackers:
    track_file = f'{BENCHMARK_DIR}/{tracker_name}/{SPLIT}/{sequences[0]}.txt'
    if os.path.exists(track_file):
        plot_trajectories(track_file, f'{tracker_name.upper()} — Trajectories on {sequences[0]}')

## 8. Summary & Next Steps

This notebook established:
- Detection baseline (RF-DETR or synthetic)
- Multi-tracker comparison framework
- TrackEval evaluation pipeline

**To add more trackers:**
```python
# Install the package
!pip install <package>

# Add adapter in src/tracking/benchmark_trackers.py
# Or for quick testing, use the standalone tracker:
from src.tracking.benchmark_trackers import TrackerAdapter, TRACKER_REGISTRY

class MyTracker(TrackerAdapter):
    name = 'my_tracker'
    def reset(self): ...
    def update(self, frame_id, dets, path=''): ...

TRACKER_REGISTRY['my_tracker'] = MyTracker
```

**Next phases:**
- Phase 1: Add homography → rink-space tracking
- Phase 2: Add jersey OCR + team classification
- Phase 3: Long-term identity fusion

See `docs/benchmark_results.md` for the full ablation table template.